# 02 — Baseline Classifier

**Goal:** establish baseline performance using *all* features, before any feature
selection is applied. Every metaheuristic notebook (GA/PSO/GWO/WOA) will be judged
against this baseline in `07_Comparison.ipynb` — if a feature-selection algorithm
can't beat (or at least match) this baseline while using fewer features, it isn't
adding value.

We evaluate two classifiers (SVM and Random Forest) using the exact same
`evaluate_subset()` function that GA/PSO/GWO/WOA will use later, so the comparison
is apples-to-apples.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from utils.preprocessing import load_processed_data
from utils.metrics import evaluate_subset, get_confusion_matrix

## 1. Load processed data (from notebook 01)

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_processed_data("../datasets")
n_features = X_train.shape[1]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Number of features:", n_features)

## 2. Full-feature mask

A mask of all 1s means "use every feature" — this is our baseline.

In [ ]:
full_mask = np.ones(n_features, dtype=int)
full_mask

## 3. Evaluate baseline: SVM

In [ ]:
svm_results = evaluate_subset(
    full_mask, X_train, X_test, y_train, y_test, classifier="svm"
)

print("SVM Baseline")
for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "n_features", "runtime"]:
    print(f"  {k}: {svm_results[k]}")

## 4. Evaluate baseline: Random Forest

In [ ]:
rf_results = evaluate_subset(
    full_mask, X_train, X_test, y_train, y_test, classifier="random_forest"
)

print("Random Forest Baseline")
for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "n_features", "runtime"]:
    print(f"  {k}: {rf_results[k]}")

## 5. Confusion matrices

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, res) in zip(axes, [("SVM", svm_results), ("Random Forest", rf_results)]):
    cm = get_confusion_matrix(y_test, res["y_pred"])
    ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 6. Save baseline results

Saved to `../results/baseline_results.csv` so `07_Comparison.ipynb` can pull it in
alongside GA/PSO/GWO/WOA results.

In [ ]:
import os
os.makedirs("../results", exist_ok=True)

baseline_df = pd.DataFrame([
    {
        "Algorithm": "Baseline (SVM)",
        "Accuracy": svm_results["accuracy"],
        "Precision": svm_results["precision"],
        "Recall": svm_results["recall"],
        "F1": svm_results["f1"],
        "ROC_AUC": svm_results["roc_auc"],
        "Features": svm_results["n_features"],
        "Runtime": svm_results["runtime"],
    },
    {
        "Algorithm": "Baseline (Random Forest)",
        "Accuracy": rf_results["accuracy"],
        "Precision": rf_results["precision"],
        "Recall": rf_results["recall"],
        "F1": rf_results["f1"],
        "ROC_AUC": rf_results["roc_auc"],
        "Features": rf_results["n_features"],
        "Runtime": rf_results["runtime"],
    },
])

baseline_df.to_csv("../results/baseline_results.csv", index=False)
baseline_df

## Next step

Proceed to `03_Genetic_Algorithm_FS.ipynb` (and then PSO, GWO, WOA). Each of those
notebooks will select a feature subset and call the exact same `evaluate_subset()`
function used here, so results are directly comparable in `07_Comparison.ipynb`.